In [20]:
import anywidget

import ipywidgets

import sys

print("Python:", sys.executable)

print("anywidget:", anywidget.__version__)

print("ipywidgets:", ipywidgets.__version__)

Python: /Applications/Blender.app/Contents/Resources/5.1/python/bin/python3.13
anywidget: 0.11.0
ipywidgets: 8.1.8


In [21]:
# Use the Gesture Recognizer widget from a JupyterLab notebook running in Blender.
#
# Run the cells (separated by `# %%`) one at a time in the Blender Jupyter kernel.
# The webcam + recognition run in the browser; the 21 hand landmarks sync into
# Blender's Python as traitlets, and a main-thread timer puppets a Blender object.

# %% 1) Make deps + the gesture_widget package importable in Blender's Python
import subprocess
import sys

REPO = "/Users/jan-hendrik/projects/caputre_motion"  # <-- adjust if you moved it

# anywidget etc. must live in BLENDER's bundled Python (this kernel), not your
# system Python. sys.executable here is Blender's python; install once.
for pkg in ("anywidget", "traitlets", "ipywidgets"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

src = f"{REPO}/src_widget"
if src not in sys.path:
    sys.path.insert(0, src)

from gesture_widget import GestureRecognizerWidget  # noqa: E402

print("gesture_widget ready in Blender's Python")


# %% 2) Show the widget, then click "Start Camera" in the output and allow webcam
w = GestureRecognizerWidget()
w.sync_interval_ms = 33  # push landmarks to Python at ~30 Hz
w


Task was destroyed but it is pending!
task: <Task pending name='Task-890' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/jan-hendrik/Library/Application Support/Blender/5.1/extensions/.local/lib/python3.13/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-891' coro=<Kernel.shell_main() running at /Users/jan-hendrik/Library/Application Support/Blender/5.1/extensions/.local/lib/python3.13/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/jan-hendrik/Library/Application Support/Blender/5.1/extensions/.local/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py:563]>
/Applications/Blender.app/Contents/Resources/5.1/python/lib/python3.13/collections/__init__.py:452: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  result = tuple_new(cls, iterable)
Task was destroyed but it is pending!
task: <Task pending name='Task-891' coro=<Kernel.shell_main() ru

gesture_widget ready in Blender's Python


In [18]:



# %% 3) Five cubes = the five fingertips, driven by bpy (main-thread safe)
import bpy  # noqa: E402

SCALE = 5.0   # world units spanned by the camera frame
CUBE = 0.15   # half-size of each fingertip cube

# Each fingertip traitlet -> a cube. (traitlet name on `w`, cube object name)
FINGERTIPS = [
    ("thumb_tip", "Tip_Thumb"),
    ("index_finger_tip", "Tip_Index"),
    ("middle_finger_tip", "Tip_Middle"),
    ("ring_finger_tip", "Tip_Ring"),
    ("pinky_tip", "Tip_Pinky"),
]


In [19]:

cubes = {}
for trait, obj_name in FINGERTIPS:
    obj = bpy.data.objects.get(obj_name)
    if obj is None:
        import bmesh

        mesh = bpy.data.meshes.new(obj_name)
        bm = bmesh.new()
        bmesh.ops.create_cube(bm, size=CUBE * 2)
        bm.to_mesh(mesh)
        bm.free()
        obj = bpy.data.objects.new(obj_name, mesh)
        bpy.context.scene.collection.objects.link(obj)
    cubes[trait] = obj


def _update_fingertips():
    # Reading traitlets (plain Python attributes) and doing bpy work is safe here
    # because timers run on Blender's main thread.
    for trait, obj in cubes.items():
        p = getattr(w, trait)  # [x, y, z] normalized 0..1, or [] when no hand
        if p:
            x, y, z = p
            # image space (x right, y DOWN) -> Blender (x right, z UP, y depth)
            obj.location = ((x - 0.5) * SCALE, z * SCALE, (0.5 - y) * SCALE)
    return 0.033  # seconds until next call (~30 Hz). Return None to stop.


if bpy.app.timers.is_registered(_update_fingertips):
    bpy.app.timers.unregister(_update_fingertips)
bpy.app.timers.register(_update_fingertips)

print("Timer running — 5 cubes now track your fingertips.")


# %% 4) Stop driving Blender
# bpy.app.timers.unregister(_update_fingertips)
# w.stop()  # turn the camera off


Timer running — 5 cubes now track your fingertips.
